In [ ]:
import os
import sys
import json
import csv
import math
import time
import copy
import random
import shutil
import hashlib
import platform
import warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from contextlib import nullcontext
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score, confusion_matrix
from tqdm import tqdm
warnings.filterwarnings("default")


In [ ]:
NOTEBOOK_NAME = "final_asl_custom_losses_convnext_yolo_repeat1_compact.ipynb"
EXPERIMENT_NAME = "final_asl_custom_losses_convnext_yolo_repeat1_compact"
TARGET_PYTHON_VERSION = "3.12.2"
SEED = 42
REPEATS = 1
REPEAT_ID = 1
REPEAT_MODE = "single_fixed_seed"
IMG_SIZE = 224
NUM_CLASSES = 4
CLASS_DIRS = ["1. Healthy", "2. BG", "3. WSSV", "4. WSSV_BG"]
CLASS_NAMES = ["Healthy", "BG", "WSSV", "WSSV_BG"]
CLASS_TO_IDX = {folder: idx for idx, folder in enumerate(CLASS_DIRS)}
EXPECTED_CLASS_COUNTS = {"1. Healthy": 403, "2. BG": 198, "3. WSSV": 328, "4. WSSV_BG": 220}
EXPECTED_TOTAL_IMAGES = 1149
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = Path(os.environ.get("SHRIMP_ASL_OUTPUT_DIR", str(PROJECT_ROOT / "final_asl_custom_losses_convnext_yolo_repeat1_compact_outputs"))).expanduser().resolve()
CHECKPOINTS_DIR = OUTPUT_DIR / "checkpoints"
YOLO_RUNS_DIR = OUTPUT_DIR / "yolo_runs"
YOLO_DATA_DIR = OUTPUT_DIR / "yolo_fixed_dataset"
FINAL_SUMMARY_XLSX_PATH = OUTPUT_DIR / "final_summary.xlsx"
FINAL_SUMMARY_CSV_PATH = OUTPUT_DIR / "final_summary.csv"
LOSS_RUN_SUMMARY_RAW_PATH = OUTPUT_DIR / "loss_run_summary_raw.csv"
RUN_AUDIT_PATH = OUTPUT_DIR / "run_audit.json"
ENVIRONMENT_VERSIONS_PATH = OUTPUT_DIR / "environment_versions.json"
FIXED_SPLIT_MANIFEST_PATH = OUTPUT_DIR / "fixed_split_manifest_seed42_with_md5.csv"
YOLO_SPLIT_MANIFEST_PATH = OUTPUT_DIR / "yolo_split_manifest_seed42_with_md5.csv"
MISSING_OR_FAILED_RUNS_PATH = OUTPUT_DIR / "missing_or_failed_runs.csv"
OUTPUT_MEMORY_SUMMARY_PATH = OUTPUT_DIR / "memory_bank_asl_custom_losses_experiment_summary.md"
for directory in [OUTPUT_DIR, CHECKPOINTS_DIR, YOLO_RUNS_DIR, YOLO_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = os.environ.get("RUN_TRAINING", "0").strip() == "1"
ALLOW_NONSTANDARD_DATASET = os.environ.get("ALLOW_NONSTANDARD_DATASET", "0").strip() == "1"
ALLOW_RANDOM_PRETRAINED_FALLBACK = os.environ.get("ALLOW_RANDOM_PRETRAINED_FALLBACK", "0").strip() == "1"
RESUME_COMPLETED = os.environ.get("RESUME_COMPLETED", "1").strip() == "1"
CONVNEXT_USE_AMP = os.environ.get("CONVNEXT_USE_AMP", "1").strip() == "1"
YOLO_MODEL_NAME = "yolo26m-cls"
YOLO_BATCH_SIZE = 128
YOLO_WORKERS = 8
YOLO_EPOCHS = 30
YOLO_PATIENCE = 15
CONVNEXT_EPOCHS = 30
CONVNEXT_PATIENCE = 5
CONVNEXT_WARMUP_EPOCHS = 5
CONVNEXT_PAPER_BATCH_SIZE = 128
CONVNEXT_MICRO_BATCH_SIZE = int(os.environ.get("CONVNEXT_MICRO_BATCH_SIZE", "32"))
CONVNEXT_ACCUMULATION_STEPS = max(1, CONVNEXT_PAPER_BATCH_SIZE // CONVNEXT_MICRO_BATCH_SIZE)
CONVNEXT_EVAL_BATCH_SIZE = 128
CONVNEXT_WORKERS = int(os.environ.get("CONVNEXT_WORKERS", "2"))
CONVNEXT_LEARNING_RATE = 1e-3
CONVNEXT_BACKBONE_FINETUNE_LR = 2e-5
CONVNEXT_HEAD_FINETUNE_LR = 1e-4
CONVNEXT_STEP_SIZE = 3
CONVNEXT_STEP_GAMMA = 0.9
CONVNEXT_UNFREEZE_FROM_FEATURE_INDEX = 5
CONVNEXT_SELECTION_METRIC = "val_macro_f1_then_val_loss"
DEFAULT_DATA_DIR = Path("/home/drnguyenvinh/notebooks/uynnhy/processed-images/processed_images")
REFERENCE_SHRIMPXNET_DATA_DIR = "/content/drive/MyDrive/shrimp_disease_images/Processed_Rembg_Images"
DATA_DIR_ENV_KEYS = ["SHRIMP_DATA_DIR", "DATA_DIR_OVERRIDE"]
def has_class_dirs(path):
    return all((path / class_dir).exists() for class_dir in CLASS_DIRS)
def resolve_data_dir():
    for key in DATA_DIR_ENV_KEYS:
        value = os.environ.get(key)
        if value:
            candidate = Path(value).expanduser().resolve()
            if has_class_dirs(candidate):
                return candidate
            raise FileNotFoundError(f"{key}={candidate} does not contain all class folders")
    candidates = [
        DEFAULT_DATA_DIR,
        PROJECT_ROOT / "uynnhy" / "processed-images" / "processed_images",
        PROJECT_ROOT / "uynnhy" / "processed-images",
        PROJECT_ROOT / "processed_images",
        PROJECT_ROOT / "processed-images" / "processed_images",
        PROJECT_ROOT / "processed-images",
    ]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if has_class_dirs(candidate):
            return candidate
    raise FileNotFoundError("Set SHRIMP_DATA_DIR to the ShrimpDiseaseImageBD V3 processed image directory")
DATA_DIR = resolve_data_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if RUN_TRAINING and DEVICE.type != "cuda":
    raise RuntimeError("RUN_TRAINING=1 requires the Linux RTX 4090 CUDA environment")
print(f"Notebook: {NOTEBOOK_NAME}")
print(f"RUN_TRAINING={RUN_TRAINING}")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"DEVICE={DEVICE}")


In [ ]:
def reset_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
def utc_now():
    return datetime.now(timezone.utc).isoformat()
def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, default=str)
def file_md5(path, chunk_size=1024 * 1024):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()
def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()
def sanitize_name(value):
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(value)).strip("_")
def ensure_under(paths, root):
    root = Path(root).resolve()
    for item in paths:
        item_path = Path(item).resolve()
        if root not in [item_path, *item_path.parents]:
            raise AssertionError(f"{item_path} is not under {root}")
def model_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 * 1024) if path.exists() else np.nan
def safe_float(value):
    try:
        if value is None:
            return np.nan
        return float(value)
    except Exception:
        return np.nan
def environment_versions():
    modules = {}
    for name in ["numpy", "pandas", "torch", "torchvision", "sklearn", "PIL", "tqdm", "ultralytics", "openpyxl"]:
        try:
            module = __import__(name)
            modules[name] = getattr(module, "__version__", "available")
        except Exception as exc:
            modules[name] = f"unavailable:{type(exc).__name__}"
    return {
        "timestamp_utc": utc_now(),
        "python_version": sys.version,
        "target_python_version": TARGET_PYTHON_VERSION,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
        "torch_cuda_version": getattr(torch.version, "cuda", None),
        "modules": modules,
    }
reset_all_seeds(SEED)
save_json(ENVIRONMENT_VERSIONS_PATH, environment_versions())


In [ ]:
def discover_images(data_dir):
    rows = []
    for class_dir in CLASS_DIRS:
        folder = data_dir / class_dir
        if not folder.exists():
            raise FileNotFoundError(str(folder))
        for path in sorted(folder.rglob("*")):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                rel_path = path.relative_to(data_dir).as_posix()
                label = CLASS_TO_IDX[class_dir]
                rows.append({"rel_path": rel_path, "path": str(path.resolve()), "class_dir": class_dir, "class_name": CLASS_NAMES[label], "label": label})
    frame = pd.DataFrame(rows).sort_values("rel_path").reset_index(drop=True)
    if frame.empty:
        raise RuntimeError(f"No images found under {data_dir}")
    return frame
def validate_dataset_counts(frame):
    total = len(frame)
    counts = frame.groupby("class_dir").size().to_dict()
    if not ALLOW_NONSTANDARD_DATASET and total != EXPECTED_TOTAL_IMAGES:
        raise AssertionError(f"Expected {EXPECTED_TOTAL_IMAGES} images, found {total}. Set ALLOW_NONSTANDARD_DATASET=1 to override")
    if not ALLOW_NONSTANDARD_DATASET:
        for class_dir, expected in EXPECTED_CLASS_COUNTS.items():
            observed = int(counts.get(class_dir, 0))
            if observed != expected:
                raise AssertionError(f"Expected {expected} images for {class_dir}, found {observed}. Set ALLOW_NONSTANDARD_DATASET=1 to override")
def add_source_md5(frame):
    frame = frame.copy()
    frame["md5"] = [file_md5(path) for path in frame["path"]]
    return frame
def make_fixed_split():
    frame = add_source_md5(discover_images(DATA_DIR))
    validate_dataset_counts(frame)
    missing = [path for path in frame["path"] if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(missing[0])
    train_frame, tmp_frame = train_test_split(frame, test_size=0.30, stratify=frame["label"], random_state=SEED, shuffle=True)
    val_frame, test_frame = train_test_split(tmp_frame, test_size=0.50, stratify=tmp_frame["label"], random_state=SEED, shuffle=True)
    split_frames = []
    for split_name, split_frame in [("train", train_frame), ("val", val_frame), ("test", test_frame)]:
        part = split_frame.copy()
        part["split"] = split_name
        split_frames.append(part)
    manifest = pd.concat(split_frames, ignore_index=True).sort_values(["split", "rel_path"]).reset_index(drop=True)
    sets = {name: set(manifest.loc[manifest["split"] == name, "rel_path"]) for name in ["train", "val", "test"]}
    if sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"]:
        raise AssertionError("Split rel_path overlap detected")
    expected_counts = {"train": 804, "val": 172, "test": 173}
    if not ALLOW_NONSTANDARD_DATASET:
        observed_counts = manifest.groupby("split").size().to_dict()
        for split_name, expected in expected_counts.items():
            if int(observed_counts.get(split_name, 0)) != expected:
                raise AssertionError(f"Expected {expected} {split_name} images, found {observed_counts.get(split_name, 0)}")
    manifest.to_csv(FIXED_SPLIT_MANIFEST_PATH, index=False)
    return manifest
def copy_yolo_split(fixed_manifest):
    if YOLO_DATA_DIR.exists():
        shutil.rmtree(YOLO_DATA_DIR)
    for split_name in ["train", "val", "test"]:
        for class_dir in CLASS_DIRS:
            (YOLO_DATA_DIR / split_name / class_dir).mkdir(parents=True, exist_ok=True)
    rows = []
    for row in fixed_manifest.itertuples(index=False):
        src = Path(row.path)
        dst = YOLO_DATA_DIR / row.split / row.class_dir / src.name
        shutil.copy2(src, dst)
        item = row._asdict()
        item["source_md5"] = row.md5
        item["yolo_path"] = str(dst.resolve())
        item["yolo_md5"] = file_md5(dst)
        rows.append(item)
    manifest = pd.DataFrame(rows)
    if not (manifest["source_md5"] == manifest["yolo_md5"]).all():
        raise AssertionError("YOLO copy MD5 mismatch")
    ensure_under(manifest["yolo_path"].tolist(), YOLO_DATA_DIR)
    if manifest["yolo_path"].duplicated().any():
        raise AssertionError("Duplicate YOLO destination path detected")
    manifest.to_csv(YOLO_SPLIT_MANIFEST_PATH, index=False)
    return manifest
fixed_manifest = make_fixed_split()
yolo_manifest = copy_yolo_split(fixed_manifest)
train_df = fixed_manifest[fixed_manifest["split"] == "train"].sort_values("rel_path").reset_index(drop=True)
val_df = fixed_manifest[fixed_manifest["split"] == "val"].sort_values("rel_path").reset_index(drop=True)
test_df = fixed_manifest[fixed_manifest["split"] == "test"].sort_values("rel_path").reset_index(drop=True)
yolo_train_df = yolo_manifest[yolo_manifest["split"] == "train"].sort_values("rel_path").reset_index(drop=True)
yolo_val_df = yolo_manifest[yolo_manifest["split"] == "val"].sort_values("rel_path").reset_index(drop=True)
yolo_test_df = yolo_manifest[yolo_manifest["split"] == "test"].sort_values("rel_path").reset_index(drop=True)
CLASS_COUNTS = [int((train_df["label"] == idx).sum()) for idx in range(NUM_CLASSES)]
FIXED_SPLIT_ID = hashlib.sha256(pd.util.hash_pandas_object(fixed_manifest[["rel_path", "split", "label", "md5"]], index=False).values.tobytes()).hexdigest()
print(fixed_manifest.groupby(["split", "class_name"]).size().unstack(fill_value=0).reindex(["train", "val", "test"])[CLASS_NAMES])
print(f"FIXED_SPLIT_ID={FIXED_SPLIT_ID}")


In [ ]:
LOSS_RUNS = [
    {"loss_key": "baseline_ce", "display_name": "CE", "category": "existing_baseline"},
    {"loss_key": "asl_single_label", "display_name": "ASLSingleLabel", "category": "existing_asl_baseline"},
    {"loss_key": "false_coinfection_cost_ce", "display_name": "False-CoInfection Cost CE", "category": "existing_custom_reference"},
    {"loss_key": "poly_dcs_ce", "display_name": "Poly-DCS-CE", "category": "existing_custom_reference"},
    {"loss_key": "sce", "display_name": "SCE", "category": "existing_robust_baseline"},
    {"loss_key": "ldam", "display_name": "LDAM", "category": "existing_margin_baseline"},
    {"loss_key": "false_coinfection_cost_asl", "display_name": "False-CoInfection Cost ASL", "category": "proposed_asl_variant"},
    {"loss_key": "poly_dcs_asl", "display_name": "Poly-DCS-ASL", "category": "proposed_asl_variant"},
    {"loss_key": "pairwise_coinfection_ranking_asl", "display_name": "Pairwise Co-Infection Ranking ASL", "category": "proposed_asl_variant"},
    {"loss_key": "confidence_gated_dcs_asl", "display_name": "Confidence-Gated DCS-ASL", "category": "proposed_asl_variant"},
    {"loss_key": "attribute_projection_asl", "display_name": "Attribute-Projection ASL", "category": "proposed_asl_variant"},
    {"loss_key": "sce_asl_hybrid", "display_name": "SCE-ASL Hybrid", "category": "proposed_asl_variant"},
    {"loss_key": "ldam_asl_hybrid", "display_name": "LDAM-ASL Hybrid", "category": "proposed_asl_variant"},
    {"loss_key": "robust_gap_asl", "display_name": "Robust Gap ASL", "category": "proposed_asl_variant"},
    {"loss_key": "distribution_balanced_attribute_asl", "display_name": "Distribution-Balanced Attribute ASL", "category": "proposed_asl_variant"},
    {"loss_key": "cost_poly_asl", "display_name": "Cost-Poly ASL", "category": "proposed_asl_variant"},
]
LOSS_LABELS = {item["loss_key"]: item["display_name"] for item in LOSS_RUNS}
LOSS_CATEGORIES = {item["loss_key"]: item["category"] for item in LOSS_RUNS}
if len(LOSS_LABELS) != 16:
    raise AssertionError("LOSS_RUNS must contain 16 unique losses")
DEFAULT_COST_MATRIX = torch.tensor([[0.0, 1.0, 1.0, 1.5], [1.0, 0.0, 1.5, 2.0], [1.0, 1.5, 0.0, 2.5], [1.5, 1.0, 1.0, 0.0]], dtype=torch.float32)
ATTRIBUTE_MATRIX = torch.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=torch.float32)
def reduce_values(values, reduction):
    if reduction == "mean":
        return values.mean()
    if reduction == "sum":
        return values.sum()
    return values
def target_probs(probs, targets):
    return probs.gather(1, targets.long().view(-1, 1)).squeeze(1)
def pairwise_single_to_mix_penalty(probs, targets, margin=0.10, smooth=False, temperature=5.0):
    p_bg = probs[:, 1]
    p_wssv = probs[:, 2]
    p_mix = probs[:, 3]
    values = torch.zeros_like(p_mix)
    bg_mask = targets == 1
    wssv_mask = targets == 2
    if bg_mask.any():
        gap = margin + p_mix[bg_mask] - p_bg[bg_mask]
        values[bg_mask] = F.softplus(temperature * gap) / temperature if smooth else F.relu(gap).pow(2)
    if wssv_mask.any():
        gap = margin + p_mix[wssv_mask] - p_wssv[wssv_mask]
        values[wssv_mask] = F.softplus(temperature * gap) / temperature if smooth else F.relu(gap).pow(2)
    return values
def weak_mix_guard_penalty(probs, targets, margin=0.05):
    p_bg = probs[:, 1]
    p_wssv = probs[:, 2]
    p_mix = probs[:, 3]
    values = torch.zeros_like(p_mix)
    mix_mask = targets == 3
    if mix_mask.any():
        strongest_single = torch.maximum(p_bg[mix_mask], p_wssv[mix_mask])
        values[mix_mask] = F.relu(margin + strongest_single - p_mix[mix_mask]).pow(2)
    return values
def manual_bce_probs(probabilities, targets):
    probabilities = probabilities.float().clamp(1e-6, 1.0 - 1e-6)
    targets = targets.float()
    return -(targets * probabilities.log() + (1.0 - targets) * (1.0 - probabilities).log())
class ASLSingleLabel(nn.Module):
    def __init__(self, gamma_pos=0, gamma_neg=4, eps=0.1, reduction="mean"):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.eps = eps
        self.reduction = reduction
        self.logsoftmax = nn.LogSoftmax(dim=-1)
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        num_classes = logits.size(-1)
        log_preds = self.logsoftmax(logits)
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
        anti_targets = 1.0 - one_hot
        xs_pos = torch.exp(log_preds) * one_hot
        xs_neg = (1.0 - torch.exp(log_preds)) * anti_targets
        asymmetric_w = torch.pow(1.0 - xs_pos - xs_neg, self.gamma_pos * one_hot + self.gamma_neg * anti_targets)
        weighted_log_preds = log_preds * asymmetric_w
        if self.eps > 0:
            one_hot = one_hot.mul(1.0 - self.eps).add(self.eps / num_classes)
        loss = -(one_hot * weighted_log_preds).sum(dim=-1)
        return reduce_values(loss, self.reduction)
class SCELoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.1, num_classes=4, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.num_classes = num_classes
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        probs = F.softmax(logits, dim=1).clamp(1e-7, 1.0)
        one_hot = F.one_hot(targets, self.num_classes).float().clamp(1e-4, 1.0)
        rce = -(probs * one_hot.log()).sum(dim=1)
        return reduce_values(self.alpha * ce + self.beta * rce, self.reduction)
class LDAMLoss(nn.Module):
    def __init__(self, class_counts, max_m=0.5, s=30.0, reduction="mean"):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        margins = 1.0 / torch.sqrt(torch.sqrt(counts.clamp_min(1.0)))
        margins = margins * (max_m / margins.max())
        self.register_buffer("margins", margins)
        self.s = s
        self.reduction = reduction
    def adjusted_logits(self, logits, targets):
        targets = targets.long().view(-1)
        margins = self.margins.to(logits.device, logits.dtype)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, targets.unsqueeze(1), True)
        target_margins = margins[targets].unsqueeze(1)
        return torch.where(index, logits - target_margins, logits)
    def forward(self, logits, targets):
        adjusted = self.adjusted_logits(logits, targets)
        return F.cross_entropy(self.s * adjusted, targets.long().view(-1), reduction=self.reduction)
class FalseCoInfectionCostCE(nn.Module):
    def __init__(self, lambda_cost=0.05, reduction="mean"):
        super().__init__()
        self.lambda_cost = lambda_cost
        self.reduction = reduction
        self.register_buffer("cost", DEFAULT_COST_MATRIX.clone())
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        probs = F.softmax(logits, dim=1)
        expected_cost = (probs * self.cost.to(logits.device, logits.dtype)[targets]).sum(dim=1)
        return reduce_values(ce + self.lambda_cost * expected_cost, self.reduction)
class PolyDCSCE(nn.Module):
    def __init__(self, epsilon=0.5, margin_single=0.10, margin_mix=0.05, lambda_single=0.05, lambda_mix=0.02, reduction="mean"):
        super().__init__()
        self.epsilon = epsilon
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        ce = F.cross_entropy(logits, targets, reduction="none")
        probs = F.softmax(logits, dim=1)
        poly = self.epsilon * (1.0 - target_probs(probs, targets))
        penalty = self.lambda_single * pairwise_single_to_mix_penalty(probs, targets, self.margin_single)
        guard = self.lambda_mix * weak_mix_guard_penalty(probs, targets, self.margin_mix)
        return reduce_values(ce + poly + penalty + guard, self.reduction)
class FalseCoInfectionCostASL(nn.Module):
    def __init__(self, lambda_cost=0.03, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.lambda_cost = lambda_cost
        self.reduction = reduction
        self.register_buffer("cost", DEFAULT_COST_MATRIX.clone())
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        probs = F.softmax(logits, dim=1)
        expected_cost = (probs * self.cost.to(logits.device, logits.dtype)[targets]).sum(dim=1)
        return reduce_values(base + self.lambda_cost * expected_cost, self.reduction)
class PolyDCSASL(nn.Module):
    def __init__(self, epsilon=0.25, margin_single=0.10, margin_mix=0.05, lambda_single=0.05, lambda_mix=0.02, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.epsilon = epsilon
        self.margin_single = margin_single
        self.margin_mix = margin_mix
        self.lambda_single = lambda_single
        self.lambda_mix = lambda_mix
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        probs = F.softmax(logits, dim=1)
        poly = self.epsilon * (1.0 - target_probs(probs, targets))
        penalty = self.lambda_single * pairwise_single_to_mix_penalty(probs, targets, self.margin_single)
        guard = self.lambda_mix * weak_mix_guard_penalty(probs, targets, self.margin_mix)
        return reduce_values(base + poly + penalty + guard, self.reduction)
class PairwiseCoInfectionRankingASL(nn.Module):
    def __init__(self, margin=0.10, lambda_rank=0.10, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.margin = margin
        self.lambda_rank = lambda_rank
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        probs = F.softmax(logits, dim=1)
        loss = self.base(logits, targets) + self.lambda_rank * pairwise_single_to_mix_penalty(probs, targets, self.margin)
        return reduce_values(loss, self.reduction)
class ConfidenceGatedDCSASL(nn.Module):
    def __init__(self, threshold=0.30, margin=0.10, lambda_penalty=0.15, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.threshold = threshold
        self.margin = margin
        self.lambda_penalty = lambda_penalty
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        probs = F.softmax(logits, dim=1)
        gate = (probs[:, 3].detach() > self.threshold).to(probs.dtype)
        directional = pairwise_single_to_mix_penalty(probs, targets, self.margin)
        return reduce_values(self.base(logits, targets) + self.lambda_penalty * gate * directional, self.reduction)
class AttributeProjectionASL(nn.Module):
    def __init__(self, lambda_attr=0.03, lambda_single=0.05, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.lambda_attr = lambda_attr
        self.lambda_single = lambda_single
        self.reduction = reduction
        self.register_buffer("attrs", ATTRIBUTE_MATRIX.clone())
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        probs = F.softmax(logits, dim=1)
        attrs = self.attrs.to(logits.device, logits.dtype)
        attr_probs = (probs @ attrs).float().clamp(1e-6, 1.0 - 1e-6)
        attr_targets = attrs[targets].float()
        attr_loss = manual_bce_probs(attr_probs, attr_targets).sum(dim=1)
        single_mask = ((targets == 1) | (targets == 2)).to(probs.dtype)
        single_penalty = single_mask * probs[:, 3].pow(2)
        return reduce_values(base + self.lambda_attr * attr_loss.to(base.dtype) + self.lambda_single * single_penalty, self.reduction)
class SCEASLHybrid(nn.Module):
    def __init__(self, alpha=1.0, beta=0.1, lambda_directional=0.05, margin=0.10, num_classes=4, reduction="mean"):
        super().__init__()
        self.asl = ASLSingleLabel(reduction="none")
        self.alpha = alpha
        self.beta = beta
        self.lambda_directional = lambda_directional
        self.margin = margin
        self.num_classes = num_classes
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        probs = F.softmax(logits, dim=1).clamp(1e-7, 1.0)
        one_hot = F.one_hot(targets, self.num_classes).float().clamp(1e-4, 1.0)
        reverse_ce = -(probs * one_hot.log()).sum(dim=1)
        directional = pairwise_single_to_mix_penalty(probs, targets, self.margin)
        return reduce_values(self.alpha * self.asl(logits, targets) + self.beta * reverse_ce + self.lambda_directional * directional, self.reduction)
class LDAMASLHybrid(nn.Module):
    def __init__(self, class_counts, max_m=0.30, s=15.0, lambda_directional=0.03, margin=0.10, reduction="mean"):
        super().__init__()
        self.ldam = LDAMLoss(class_counts, max_m=max_m, s=s, reduction="none")
        self.asl = ASLSingleLabel(reduction="none")
        self.s = s
        self.lambda_directional = lambda_directional
        self.margin = margin
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        adjusted = self.ldam.adjusted_logits(logits, targets)
        probs = F.softmax(logits, dim=1)
        directional = pairwise_single_to_mix_penalty(probs, targets, self.margin)
        return reduce_values(self.asl(self.s * adjusted, targets) + self.lambda_directional * directional, self.reduction)
class RobustGapASL(nn.Module):
    def __init__(self, margin=0.10, lambda_gap=0.05, temperature=5.0, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.margin = margin
        self.lambda_gap = lambda_gap
        self.temperature = temperature
        self.reduction = reduction
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        probs = F.softmax(logits, dim=1)
        gap = pairwise_single_to_mix_penalty(probs, targets, self.margin, smooth=True, temperature=self.temperature)
        return reduce_values(self.base(logits, targets) + self.lambda_gap * gap, self.reduction)
class DistributionBalancedAttributeASL(nn.Module):
    def __init__(self, class_counts, lambda_attr=0.03, lambda_single=0.05, beta=0.999, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.lambda_attr = lambda_attr
        self.lambda_single = lambda_single
        self.beta = beta
        self.reduction = reduction
        self.register_buffer("attrs", ATTRIBUTE_MATRIX.clone())
        counts = torch.tensor(class_counts, dtype=torch.float32)
        bg_present = counts[1] + counts[3]
        wssv_present = counts[2] + counts[3]
        total = counts.sum()
        pos_counts = torch.tensor([bg_present, wssv_present], dtype=torch.float32).clamp_min(1.0)
        neg_counts = (total - pos_counts).clamp_min(1.0)
        pos_w = (1.0 - beta) / (1.0 - torch.pow(torch.tensor(beta, dtype=torch.float32), pos_counts))
        neg_w = (1.0 - beta) / (1.0 - torch.pow(torch.tensor(beta, dtype=torch.float32), neg_counts))
        weights = torch.stack([pos_w, neg_w], dim=0)
        weights = weights / weights.mean()
        self.register_buffer("pos_attr_weight", weights[0])
        self.register_buffer("neg_attr_weight", weights[1])
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        base = self.base(logits, targets)
        probs = F.softmax(logits, dim=1)
        attrs = self.attrs.to(logits.device, logits.dtype)
        attr_targets = attrs[targets].float()
        attr_probs = (probs @ attrs).float().clamp(1e-6, 1.0 - 1e-6)
        attr_weights = attr_targets * self.pos_attr_weight.to(logits.device).float() + (1.0 - attr_targets) * self.neg_attr_weight.to(logits.device).float()
        attr_loss = (manual_bce_probs(attr_probs, attr_targets) * attr_weights).sum(dim=1)
        single_mask = ((targets == 1) | (targets == 2)).to(probs.dtype)
        single_penalty = single_mask * probs[:, 3].pow(2)
        return reduce_values(base + self.lambda_attr * attr_loss.to(base.dtype) + self.lambda_single * single_penalty, self.reduction)
class CostPolyASL(nn.Module):
    def __init__(self, lambda_cost=0.03, epsilon=0.25, reduction="mean"):
        super().__init__()
        self.base = ASLSingleLabel(reduction="none")
        self.lambda_cost = lambda_cost
        self.epsilon = epsilon
        self.reduction = reduction
        self.register_buffer("cost", DEFAULT_COST_MATRIX.clone())
    def forward(self, logits, targets):
        targets = targets.long().view(-1)
        probs = F.softmax(logits, dim=1)
        expected_cost = (probs * self.cost.to(logits.device, logits.dtype)[targets]).sum(dim=1)
        poly = self.epsilon * (1.0 - target_probs(probs, targets))
        return reduce_values(self.base(logits, targets) + self.lambda_cost * expected_cost + poly, self.reduction)
def make_loss(loss_key, class_counts, num_classes=4):
    if loss_key == "baseline_ce":
        return nn.CrossEntropyLoss()
    if loss_key == "asl_single_label":
        return ASLSingleLabel(gamma_pos=0, gamma_neg=4, eps=0.1)
    if loss_key == "false_coinfection_cost_ce":
        return FalseCoInfectionCostCE(lambda_cost=0.05)
    if loss_key == "poly_dcs_ce":
        return PolyDCSCE(epsilon=0.5)
    if loss_key == "sce":
        return SCELoss(alpha=1.0, beta=0.1, num_classes=num_classes)
    if loss_key == "ldam":
        return LDAMLoss(class_counts, max_m=0.5, s=30.0)
    if loss_key == "false_coinfection_cost_asl":
        return FalseCoInfectionCostASL(lambda_cost=0.03)
    if loss_key == "poly_dcs_asl":
        return PolyDCSASL(epsilon=0.25, lambda_single=0.05, lambda_mix=0.02)
    if loss_key == "pairwise_coinfection_ranking_asl":
        return PairwiseCoInfectionRankingASL(margin=0.10, lambda_rank=0.10)
    if loss_key == "confidence_gated_dcs_asl":
        return ConfidenceGatedDCSASL(threshold=0.30, margin=0.10, lambda_penalty=0.15)
    if loss_key == "attribute_projection_asl":
        return AttributeProjectionASL(lambda_attr=0.03, lambda_single=0.05)
    if loss_key == "sce_asl_hybrid":
        return SCEASLHybrid(alpha=1.0, beta=0.1, lambda_directional=0.05, margin=0.10, num_classes=num_classes)
    if loss_key == "ldam_asl_hybrid":
        return LDAMASLHybrid(class_counts, max_m=0.30, s=15.0, lambda_directional=0.03, margin=0.10)
    if loss_key == "robust_gap_asl":
        return RobustGapASL(margin=0.10, lambda_gap=0.05, temperature=5.0)
    if loss_key == "distribution_balanced_attribute_asl":
        return DistributionBalancedAttributeASL(class_counts, lambda_attr=0.03, lambda_single=0.05, beta=0.999)
    if loss_key == "cost_poly_asl":
        return CostPolyASL(lambda_cost=0.03, epsilon=0.25)
    raise KeyError(loss_key)


In [ ]:
loss_sanity = []
test_logits = torch.tensor([[2.0, 0.0, -1.0, -2.0], [0.1, 2.0, -0.5, 0.3], [0.2, -0.4, 2.2, 0.7], [-1.0, 0.4, 0.5, 2.0], [1.0, 0.1, 0.2, 0.3], [0.0, 1.0, 0.3, 0.8], [0.0, 0.4, 1.2, 0.9], [0.2, 0.5, 0.7, 1.4]], dtype=torch.float32)
test_targets = torch.tensor([0, 1, 2, 3, 0, 1, 2, 3], dtype=torch.long)
for item in LOSS_RUNS:
    loss_key = item["loss_key"]
    loss_module = make_loss(loss_key, CLASS_COUNTS, NUM_CLASSES)
    value = loss_module(test_logits, test_targets)
    if value.ndim != 0:
        raise AssertionError(f"{loss_key} did not return scalar")
    if not torch.isfinite(value):
        raise AssertionError(f"{loss_key} returned non-finite loss")
    loss_sanity.append({"loss_key": loss_key, "loss": LOSS_LABELS[loss_key], "finite_scalar": True, "value": float(value.detach().cpu())})
print(pd.DataFrame(loss_sanity))


In [ ]:
FINAL_SUMMARY_COLUMNS = ["Model Key", "Model", "Backend", "Loss Key", "Loss", "Loss Category", "Status", "Error", "Seed", "Repeat", "Test Accuracy", "Test Macro Precision", "Test Macro Recall", "Test Macro F1", "Cohen Kappa", "Healthy Recall", "BG Recall", "WSSV Recall", "WSSV_BG Recall", "BG to WSSV_BG", "WSSV to WSSV_BG", "WSSV_BG to BG", "WSSV_BG to WSSV", "Delta Macro F1 vs ASL", "Delta Kappa vs ASL", "Delta WSSV to WSSV_BG vs ASL", "Delta Macro F1 vs CE", "Best Epoch", "Best Val Macro F1", "Best Val Loss", "Latency ms/image", "FPS", "Output Dir"]
MODEL_RUNS = [
    {"model_key": "convnext_tiny_shrimpxnet", "model": "ConvNeXt-Tiny ShrimpXNet-style baseline", "backend": "torchvision"},
    {"model_key": "yolo26m_cls", "model": "YOLOv26m-cls", "backend": "ultralytics"},
]
def metric_values(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    return {
        "Test Accuracy": float(accuracy_score(y_true, y_pred)),
        "Test Macro Precision": float(macro_precision),
        "Test Macro Recall": float(macro_recall),
        "Test Macro F1": float(macro_f1),
        "Cohen Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "Healthy Recall": float(recall[0]),
        "BG Recall": float(recall[1]),
        "WSSV Recall": float(recall[2]),
        "WSSV_BG Recall": float(recall[3]),
        "BG to WSSV_BG": int(cm[1, 3]),
        "WSSV to WSSV_BG": int(cm[2, 3]),
        "WSSV_BG to BG": int(cm[3, 1]),
        "WSSV_BG to WSSV": int(cm[3, 2]),
    }
def base_row(model_key, model, backend, loss_key, status, error=""):
    return {
        "Model Key": model_key,
        "Model": model,
        "Backend": backend,
        "Loss Key": loss_key,
        "Loss": LOSS_LABELS[loss_key],
        "Loss Category": LOSS_CATEGORIES[loss_key],
        "Status": status,
        "Error": error,
        "Seed": SEED,
        "Repeat": REPEAT_ID,
        "Output Dir": str(OUTPUT_DIR),
    }
def completed_row(model_key, model, backend, loss_key, test_metrics, best_epoch, best_val_macro_f1, best_val_loss, latency_ms, fps):
    row = base_row(model_key, model, backend, loss_key, "completed")
    row.update(test_metrics)
    row.update({"Best Epoch": best_epoch, "Best Val Macro F1": best_val_macro_f1, "Best Val Loss": best_val_loss, "Latency ms/image": latency_ms, "FPS": fps})
    return row
def failed_row(model_key, model, backend, loss_key, exc):
    return base_row(model_key, model, backend, loss_key, "failed", f"{type(exc).__name__}: {exc}")
def planned_rows(default_status):
    rows = []
    for model_item in MODEL_RUNS:
        for loss_item in LOSS_RUNS:
            rows.append(base_row(model_item["model_key"], model_item["model"], model_item["backend"], loss_item["loss_key"], default_status))
    return rows
def normalize_summary_frame(frame):
    for column in FINAL_SUMMARY_COLUMNS:
        if column not in frame.columns:
            frame[column] = np.nan
    return frame[FINAL_SUMMARY_COLUMNS].copy()
def merge_with_plan(run_rows, default_status):
    by_key = {}
    for row in run_rows:
        key = (row.get("Model Key"), row.get("Loss Key"), int(row.get("Seed", SEED)), int(row.get("Repeat", REPEAT_ID)))
        if row.get("Status") != "pending":
            by_key[key] = row
    merged = []
    for row in planned_rows(default_status):
        key = (row["Model Key"], row["Loss Key"], row["Seed"], row["Repeat"])
        merged.append(by_key.get(key, row))
    return merged
def apply_deltas(frame):
    frame = frame.copy()
    for model_key in frame["Model Key"].dropna().unique():
        mask = frame["Model Key"] == model_key
        model_frame = frame[mask]
        asl = model_frame[(model_frame["Loss Key"] == "asl_single_label") & (model_frame["Status"] == "completed")]
        ce = model_frame[(model_frame["Loss Key"] == "baseline_ce") & (model_frame["Status"] == "completed")]
        if len(asl) == 1:
            asl_f1 = safe_float(asl.iloc[0]["Test Macro F1"])
            asl_kappa = safe_float(asl.iloc[0]["Cohen Kappa"])
            asl_wssv_mix = safe_float(asl.iloc[0]["WSSV to WSSV_BG"])
            frame.loc[mask, "Delta Macro F1 vs ASL"] = pd.to_numeric(frame.loc[mask, "Test Macro F1"], errors="coerce") - asl_f1
            frame.loc[mask, "Delta Kappa vs ASL"] = pd.to_numeric(frame.loc[mask, "Cohen Kappa"], errors="coerce") - asl_kappa
            frame.loc[mask, "Delta WSSV to WSSV_BG vs ASL"] = pd.to_numeric(frame.loc[mask, "WSSV to WSSV_BG"], errors="coerce") - asl_wssv_mix
        if len(ce) == 1:
            ce_f1 = safe_float(ce.iloc[0]["Test Macro F1"])
            frame.loc[mask, "Delta Macro F1 vs CE"] = pd.to_numeric(frame.loc[mask, "Test Macro F1"], errors="coerce") - ce_f1
    return frame
def save_output_memory_summary(summary_frame):
    completed = summary_frame[summary_frame["Status"] == "completed"].copy()
    lines = []
    lines.extend(["ASL custom losses compact experiment summary", "", f"Timestamp UTC: {utc_now()}", f"Output directory: {OUTPUT_DIR}", f"Completed runs: {len(completed)} of {len(summary_frame)}", ""])
    if not completed.empty:
        ranked = completed.sort_values(["Test Macro F1", "Cohen Kappa", "WSSV_BG Recall"], ascending=[False, False, False])
        best = ranked.iloc[0].to_dict()
        lines.extend(["Current best completed row", "", f"Model: {best.get('Model')}", f"Loss: {best.get('Loss')}", f"Test Macro F1: {best.get('Test Macro F1')}", f"Cohen Kappa: {best.get('Cohen Kappa')}", ""])
    lines.extend(["Interpretation guardrails", "", "ASLSingleLabel is an existing official-style ASL baseline, not a proposed method.", "The custom variants are dataset-specific ASL-based co-infection suppression variants.", "Do not claim improvement until final_summary.xlsx and final_summary.csv contain completed rows for the relevant comparisons."])
    OUTPUT_MEMORY_SUMMARY_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
def save_summaries(run_rows, default_status=None):
    if default_status is None:
        default_status = "pending" if RUN_TRAINING else "not_run_training_disabled"
    merged = merge_with_plan(run_rows, default_status)
    raw_frame = normalize_summary_frame(pd.DataFrame(merged))
    summary_frame = apply_deltas(raw_frame)
    raw_frame.to_csv(LOSS_RUN_SUMMARY_RAW_PATH, index=False)
    summary_frame.to_csv(FINAL_SUMMARY_CSV_PATH, index=False)
    with pd.ExcelWriter(FINAL_SUMMARY_XLSX_PATH, engine="openpyxl") as writer:
        summary_frame.to_excel(writer, sheet_name="final_summary", index=False)
        raw_frame.to_excel(writer, sheet_name="raw_runs", index=False)
        pd.DataFrame(LOSS_RUNS).to_excel(writer, sheet_name="loss_config", index=False)
        pd.DataFrame(MODEL_RUNS).to_excel(writer, sheet_name="model_config", index=False)
        fixed_manifest.groupby(["split", "class_name"]).size().reset_index(name="count").to_excel(writer, sheet_name="split_counts", index=False)
    missing = summary_frame[summary_frame["Status"] != "completed"].copy()
    missing.to_csv(MISSING_OR_FAILED_RUNS_PATH, index=False)
    save_output_memory_summary(summary_frame)
    return summary_frame
def load_existing_rows():
    if not RESUME_COMPLETED or not LOSS_RUN_SUMMARY_RAW_PATH.exists():
        return []
    frame = pd.read_csv(LOSS_RUN_SUMMARY_RAW_PATH)
    allowed_models = {item["model_key"] for item in MODEL_RUNS}
    allowed_losses = {item["loss_key"] for item in LOSS_RUNS}
    frame = frame[(frame["Model Key"].isin(allowed_models)) & (frame["Loss Key"].isin(allowed_losses)) & (frame["Seed"] == SEED) & (frame["Repeat"] == REPEAT_ID)]
    frame = frame[frame["Status"].isin(["completed", "failed"])]
    return frame.to_dict("records")
def replace_run_row(run_rows, row):
    key = (row["Model Key"], row["Loss Key"], int(row["Seed"]), int(row["Repeat"]))
    kept = [item for item in run_rows if (item.get("Model Key"), item.get("Loss Key"), int(item.get("Seed", SEED)), int(item.get("Repeat", REPEAT_ID))) != key]
    kept.append(row)
    return kept


In [ ]:
weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.90, 1.10), interpolation=InterpolationMode.BICUBIC, antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10, interpolation=InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
eval_transform = weights.transforms()
class ShrimpManifestDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.sort_values("rel_path").reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row["label"]), row["rel_path"]
class ShrimpXNet(nn.Module):
    def __init__(self, pretrained_weights):
        super().__init__()
        self.backbone = models.convnext_tiny(weights=pretrained_weights)
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False
        self.features = self.backbone.features
        self.avgpool = self.backbone.avgpool
        self.norm = self.backbone.classifier[0]
        num_features = self.backbone.classifier[2].in_features
        self.classifier = nn.Sequential(nn.Linear(num_features, 512), nn.ReLU(inplace=True), nn.Dropout(p=0.5), nn.Linear(512, NUM_CLASSES))
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.norm(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)
    def unfreeze_finetune_layers(self, from_feature_index=CONVNEXT_UNFREEZE_FROM_FEATURE_INDEX):
        for module in self.features[from_feature_index:]:
            for parameter in module.parameters():
                parameter.requires_grad = True
        for parameter in self.norm.parameters():
            parameter.requires_grad = True
def create_shrimpxnet_model():
    try:
        model = ShrimpXNet(weights).to(DEVICE)
        return model, True
    except Exception as exc:
        if not ALLOW_RANDOM_PRETRAINED_FALLBACK:
            raise RuntimeError(f"ConvNeXt-Tiny pretrained weights unavailable and ALLOW_RANDOM_PRETRAINED_FALLBACK=0: {exc}") from exc
        model = ShrimpXNet(None).to(DEVICE)
        return model, False
def convnext_autocast():
    enabled = CONVNEXT_USE_AMP and DEVICE.type == "cuda"
    return torch.amp.autocast("cuda", dtype=torch.float16, enabled=enabled) if enabled else nullcontext()
def make_grad_scaler():
    enabled = CONVNEXT_USE_AMP and DEVICE.type == "cuda"
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)
def make_finetune_optimizer(model):
    backbone_params = []
    head_params = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("classifier"):
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)
    return optim.Adam([{"params": backbone_params, "lr": CONVNEXT_BACKBONE_FINETUNE_LR}, {"params": head_params, "lr": CONVNEXT_HEAD_FINETUNE_LR}])
def convnext_loaders():
    train_dataset = ShrimpManifestDataset(train_df, train_transform)
    val_dataset = ShrimpManifestDataset(val_df, eval_transform)
    test_dataset = ShrimpManifestDataset(test_df, eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=CONVNEXT_MICRO_BATCH_SIZE, shuffle=True, num_workers=CONVNEXT_WORKERS, pin_memory=DEVICE.type == "cuda")
    val_loader = DataLoader(val_dataset, batch_size=CONVNEXT_EVAL_BATCH_SIZE, shuffle=False, num_workers=CONVNEXT_WORKERS, pin_memory=DEVICE.type == "cuda")
    test_loader = DataLoader(test_dataset, batch_size=CONVNEXT_EVAL_BATCH_SIZE, shuffle=False, num_workers=CONVNEXT_WORKERS, pin_memory=DEVICE.type == "cuda")
    return train_loader, val_loader, test_loader
def evaluate_convnext_loader(model, loader, criterion, timed=False):
    model.eval()
    labels_all = []
    preds_all = []
    total_loss = 0.0
    total_batches = 0
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.time() if timed else None
    with torch.no_grad():
        for images, labels, rel_paths in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with convnext_autocast():
                logits = model(images)
                loss = criterion(logits.float(), labels)
            total_loss += float(loss.detach().cpu())
            total_batches += 1
            labels_all.extend(labels.detach().cpu().numpy().astype(int).tolist())
            preds_all.extend(torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int).tolist())
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else np.nan
    metrics = metric_values(labels_all, preds_all)
    metrics["loss"] = total_loss / max(1, total_batches)
    metrics["elapsed"] = elapsed
    metrics["num_images"] = len(labels_all)
    return metrics
def train_convnext_model(loss_key):
    reset_all_seeds(SEED)
    train_loader, val_loader, test_loader = convnext_loaders()
    model, pretrained = create_shrimpxnet_model()
    criterion = make_loss(loss_key, CLASS_COUNTS, NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(filter(lambda parameter: parameter.requires_grad, model.parameters()), lr=CONVNEXT_LEARNING_RATE)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=CONVNEXT_STEP_SIZE, gamma=CONVNEXT_STEP_GAMMA)
    scaler = make_grad_scaler()
    best_state = None
    best_epoch = 0
    best_val_macro_f1 = -1.0
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    checkpoint_path = CHECKPOINTS_DIR / f"best_convnext_tiny_shrimpxnet_{loss_key}_repeat01.pth"
    for epoch in range(CONVNEXT_EPOCHS):
        if epoch == CONVNEXT_WARMUP_EPOCHS:
            model.unfreeze_finetune_layers()
            optimizer = make_finetune_optimizer(model)
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=CONVNEXT_STEP_SIZE, gamma=CONVNEXT_STEP_GAMMA)
            scaler = make_grad_scaler()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for step, (images, labels, rel_paths) in enumerate(tqdm(train_loader, desc=f"convnext {loss_key} epoch {epoch + 1}/{CONVNEXT_EPOCHS}", leave=False)):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with convnext_autocast():
                logits = model(images)
                loss = criterion(logits.float(), labels)
            scaled_loss = loss / CONVNEXT_ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            if (step + 1) % CONVNEXT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        val_metrics = evaluate_convnext_loader(model, val_loader, criterion, timed=False)
        current_f1 = safe_float(val_metrics["Test Macro F1"])
        current_loss = safe_float(val_metrics["loss"])
        improved = current_f1 > best_val_macro_f1 + 1e-12 or (abs(current_f1 - best_val_macro_f1) <= 1e-12 and current_loss < best_val_loss)
        if improved:
            best_val_macro_f1 = current_f1
            best_val_loss = current_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_state_dict": best_state, "loss_key": loss_key, "pretrained": pretrained, "epoch": best_epoch, "best_val_macro_f1": best_val_macro_f1, "best_val_loss": best_val_loss, "fixed_split_id": FIXED_SPLIT_ID}, checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= CONVNEXT_PATIENCE:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate_convnext_loader(model, test_loader, criterion, timed=True)
    elapsed = safe_float(test_metrics.pop("elapsed"))
    num_images = int(test_metrics.pop("num_images"))
    test_metrics.pop("loss", None)
    latency_ms = elapsed * 1000.0 / max(1, num_images)
    fps = num_images / elapsed if elapsed and elapsed > 0 else np.nan
    row = completed_row("convnext_tiny_shrimpxnet", "ConvNeXt-Tiny ShrimpXNet-style baseline", "torchvision", loss_key, test_metrics, best_epoch, best_val_macro_f1, best_val_loss, latency_ms, fps)
    row["Checkpoint Path"] = str(checkpoint_path)
    row["Checkpoint SHA256"] = file_sha256(checkpoint_path) if checkpoint_path.exists() else ""
    row["Pretrained"] = pretrained
    return row


In [ ]:
ULTRALYTICS_IMPORT_ERROR = ""
try:
    from ultralytics import YOLO
    try:
        from ultralytics.models.yolo.classify.train import ClassificationTrainer
    except Exception:
        from ultralytics.models.yolo.classify import ClassificationTrainer
    from ultralytics.nn.tasks import ClassificationModel
    ULTRALYTICS_AVAILABLE = True
except Exception as exc:
    ULTRALYTICS_AVAILABLE = False
    ULTRALYTICS_IMPORT_ERROR = f"{type(exc).__name__}: {exc}"
def yolo_device_arg():
    return 0 if torch.cuda.is_available() else "cpu"
BASE_YOLO_TRAIN_KWARGS = dict(data=str(YOLO_DATA_DIR), task="classify", imgsz=IMG_SIZE, epochs=YOLO_EPOCHS, batch=YOLO_BATCH_SIZE, patience=YOLO_PATIENCE, project=str(YOLO_RUNS_DIR), exist_ok=True, device=yolo_device_arg(), verbose=True, workers=YOLO_WORKERS, amp=True, optimizer="AdamW", lr0=1.25e-3, lrf=0.01, cos_lr=True, cache=True, plots=False)
ACTIVE_YOLO_LOSS_KEY = "baseline_ce"
if ULTRALYTICS_AVAILABLE:
    class LossAblationClassificationLoss:
        def __init__(self, model):
            self.loss_key = getattr(model, "loss_key", ACTIVE_YOLO_LOSS_KEY)
            self.loss_fcn = make_loss(self.loss_key, CLASS_COUNTS, NUM_CLASSES)
        def extract_logits(self, preds, targets):
            if torch.is_tensor(preds):
                return preds
            if isinstance(preds, (list, tuple)):
                if len(preds) > 1 and torch.is_tensor(preds[1]) and preds[1].ndim == 2 and preds[1].shape[0] == targets.shape[0]:
                    return preds[1]
                for item in preds:
                    if torch.is_tensor(item) and item.ndim == 2 and item.shape[0] == targets.shape[0]:
                        return item
                for item in preds:
                    if torch.is_tensor(item):
                        return item
            raise TypeError(type(preds).__name__)
        def __call__(self, preds, batch):
            targets = batch["cls"].long().view(-1)
            logits = self.extract_logits(preds, targets).to(targets.device)
            targets = targets.to(logits.device)
            if isinstance(self.loss_fcn, nn.Module):
                self.loss_fcn = self.loss_fcn.to(logits.device)
            loss = self.loss_fcn(logits.float(), targets)
            return loss, loss.detach()
    class LossAblationClassificationModel(ClassificationModel):
        def init_criterion(self):
            return LossAblationClassificationLoss(self)
    class LossAblationClassificationTrainer(ClassificationTrainer):
        def get_model(self, cfg=None, weights=None, verbose=True):
            nc = self.data["nc"] if isinstance(self.data, dict) and "nc" in self.data else NUM_CLASSES
            try:
                model = LossAblationClassificationModel(cfg, nc=nc, verbose=verbose)
            except TypeError:
                model = LossAblationClassificationModel(cfg, ch=3, nc=nc, verbose=verbose)
            model.loss_key = ACTIVE_YOLO_LOSS_KEY
            if weights:
                model.load(weights)
            return model
def evaluate_yolo_model(yolo_model, eval_frame, timed=False):
    source_paths = eval_frame["yolo_path"].tolist()
    ensure_under(source_paths, YOLO_DATA_DIR)
    missing = [path for path in source_paths if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(missing[0])
    if timed:
        _ = yolo_model.predict(source=source_paths[:1], imgsz=IMG_SIZE, device=yolo_device_arg(), verbose=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
    else:
        start = None
    predictions = yolo_model.predict(source=source_paths, imgsz=IMG_SIZE, batch=YOLO_BATCH_SIZE, device=yolo_device_arg(), verbose=False)
    if timed and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - start if timed else np.nan
    y_true = eval_frame["label"].astype(int).tolist()
    y_pred = [int(result.probs.top1) for result in predictions]
    metrics = metric_values(y_true, y_pred)
    metrics["elapsed"] = elapsed
    metrics["num_images"] = len(y_true)
    return metrics
def train_yolo_model(loss_key):
    global ACTIVE_YOLO_LOSS_KEY
    if not ULTRALYTICS_AVAILABLE:
        raise RuntimeError(f"Ultralytics unavailable: {ULTRALYTICS_IMPORT_ERROR}")
    reset_all_seeds(SEED)
    ACTIVE_YOLO_LOSS_KEY = loss_key
    run_name = sanitize_name(f"repeat01_{YOLO_MODEL_NAME}_{loss_key}")
    run_dir = YOLO_RUNS_DIR / run_name
    if run_dir.exists():
        shutil.rmtree(run_dir)
    weights_name = f"{YOLO_MODEL_NAME}.pt"
    yolo = YOLO(weights_name)
    train_kwargs = dict(BASE_YOLO_TRAIN_KWARGS)
    train_kwargs.update(seed=SEED, name=run_name)
    if loss_key == "baseline_ce":
        yolo.train(**train_kwargs)
    else:
        yolo.train(trainer=LossAblationClassificationTrainer, **train_kwargs)
    best_path = run_dir / "weights" / "best.pt"
    if not best_path.exists():
        candidates = sorted(run_dir.glob("**/best.pt"))
        if not candidates:
            raise FileNotFoundError(str(best_path))
        best_path = candidates[-1]
    best_yolo = YOLO(str(best_path))
    val_metrics = evaluate_yolo_model(best_yolo, yolo_val_df, timed=False)
    test_metrics = evaluate_yolo_model(best_yolo, yolo_test_df, timed=True)
    elapsed = safe_float(test_metrics.pop("elapsed"))
    num_images = int(test_metrics.pop("num_images"))
    best_val_macro_f1 = safe_float(val_metrics["Test Macro F1"])
    best_val_loss = np.nan
    latency_ms = elapsed * 1000.0 / max(1, num_images)
    fps = num_images / elapsed if elapsed and elapsed > 0 else np.nan
    row = completed_row("yolo26m_cls", "YOLOv26m-cls", "ultralytics", loss_key, test_metrics, np.nan, best_val_macro_f1, best_val_loss, latency_ms, fps)
    row["Checkpoint Path"] = str(best_path)
    row["Checkpoint SHA256"] = file_sha256(best_path) if best_path.exists() else ""
    row["Model Size MB"] = model_size_mb(best_path)
    return row


In [ ]:
def build_run_audit(stage):
    return {
        "stage": stage,
        "timestamp_utc": utc_now(),
        "notebook": NOTEBOOK_NAME,
        "experiment": EXPERIMENT_NAME,
        "run_training": RUN_TRAINING,
        "seed": SEED,
        "repeat_count": REPEATS,
        "target_python_version": TARGET_PYTHON_VERSION,
        "data_dir": str(DATA_DIR),
        "reference_shrimpxnet_data_dir": REFERENCE_SHRIMPXNET_DATA_DIR,
        "output_dir": str(OUTPUT_DIR),
        "fixed_split_id": FIXED_SPLIT_ID,
        "class_dirs": CLASS_DIRS,
        "class_names": CLASS_NAMES,
        "class_counts_full_expected": EXPECTED_CLASS_COUNTS,
        "class_counts_train": CLASS_COUNTS,
        "loss_runs": LOSS_RUNS,
        "models": MODEL_RUNS,
        "convnext_config": {
            "source_notebook": "experiment/shrimpxnet.ipynb",
            "model_source": "torchvision.models.convnext_tiny",
            "weights": "ConvNeXt_Tiny_Weights.IMAGENET1K_V1",
            "img_size": IMG_SIZE,
            "epochs": CONVNEXT_EPOCHS,
            "patience": CONVNEXT_PATIENCE,
            "paper_batch_size": CONVNEXT_PAPER_BATCH_SIZE,
            "micro_batch_size": CONVNEXT_MICRO_BATCH_SIZE,
            "accumulation_steps": CONVNEXT_ACCUMULATION_STEPS,
            "workers": CONVNEXT_WORKERS,
            "warmup_epochs": CONVNEXT_WARMUP_EPOCHS,
            "warmup_lr": CONVNEXT_LEARNING_RATE,
            "backbone_finetune_lr": CONVNEXT_BACKBONE_FINETUNE_LR,
            "head_finetune_lr": CONVNEXT_HEAD_FINETUNE_LR,
            "scheduler": "StepLR",
            "step_size": CONVNEXT_STEP_SIZE,
            "step_gamma": CONVNEXT_STEP_GAMMA,
            "selection_metric": CONVNEXT_SELECTION_METRIC,
            "reference_selection_metric": "val_loss",
            "amp_enabled": CONVNEXT_USE_AMP,
            "random_pretrained_fallback_allowed": ALLOW_RANDOM_PRETRAINED_FALLBACK,
        },
        "yolo_config": BASE_YOLO_TRAIN_KWARGS,
        "bug_prevention": {
            "attribute_projection_bce": "manual float32 BCE on projected probabilities",
            "path_empty_string": "no prediction path collection is used in compact output",
            "failed_rows": "Status column controls completed counts",
            "summary_filter": "current output_dir, model keys, loss keys, seed 42, repeat 1",
            "yolo_eval_paths": "yolo_path only after MD5 equality verification",
            "pretrained_fallback": "random fallback requires ALLOW_RANDOM_PRETRAINED_FALLBACK=1",
        },
    }
save_json(RUN_AUDIT_PATH, build_run_audit("before_run_loop"))
run_rows = load_existing_rows()
if not RUN_TRAINING:
    final_summary = save_summaries(run_rows, default_status="not_run_training_disabled")
else:
    for model_item in MODEL_RUNS:
        for loss_item in LOSS_RUNS:
            model_key = model_item["model_key"]
            loss_key = loss_item["loss_key"]
            already_done = any(row.get("Model Key") == model_key and row.get("Loss Key") == loss_key and row.get("Status") == "completed" for row in run_rows)
            if already_done and RESUME_COMPLETED:
                continue
            try:
                if model_key == "convnext_tiny_shrimpxnet":
                    row = train_convnext_model(loss_key)
                elif model_key == "yolo26m_cls":
                    row = train_yolo_model(loss_key)
                else:
                    raise KeyError(model_key)
            except Exception as exc:
                row = failed_row(model_key, model_item["model"], model_item["backend"], loss_key, exc)
            run_rows = replace_run_row(run_rows, row)
            final_summary = save_summaries(run_rows, default_status="pending")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
final_summary = save_summaries(run_rows, default_status="pending" if RUN_TRAINING else "not_run_training_disabled")
save_json(RUN_AUDIT_PATH, build_run_audit("after_run_loop"))
print(final_summary.sort_values(["Status", "Model Key", "Loss Key"]).to_string(index=False))
print(f"final_summary_csv={FINAL_SUMMARY_CSV_PATH}")
print(f"final_summary_xlsx={FINAL_SUMMARY_XLSX_PATH}")
